# Combined Test: Subject-Holdout + Dateline-Stripped — ISOT

This version strips the dateline AND holds out subjects, isolating whether any genuine signal exists once both known shortcuts are removed.

- **Train on:** politicsNews, News, left-news, politics — dateline stripped
- **Test on:** worldnews, US_News, Government News, Middle-east — dateline stripped

## 1. Load Data, Strip Dateline, Build Subject-Holdout Split


In [1]:
import pandas as pd
import re

train_df_orig = pd.read_csv("train.csv")
val_df_orig = pd.read_csv("val.csv")
test_df_orig = pd.read_csv("test.csv")

full_df = pd.concat([train_df_orig, val_df_orig, test_df_orig], ignore_index=True)

def strip_dateline(text, check_chars=150):
    pattern = r"^.{0,80}\(Reuters\)\s*[-–—]\s*"
    stripped = re.sub(pattern, "", str(text)[:check_chars], count=1) + str(text)[check_chars:]
    return stripped

full_df["text"] = full_df["text"].apply(strip_dateline)

TRAIN_SUBJECTS = ["politicsNews", "News", "left-news", "politics"]
TEST_SUBJECTS = ["worldnews", "US_News", "Government News", "Middle-east"]

holdout_train_df = full_df[full_df["subject"].isin(TRAIN_SUBJECTS)].reset_index(drop=True)
holdout_test_df = full_df[full_df["subject"].isin(TEST_SUBJECTS)].reset_index(drop=True)

print(f"Holdout train set: {len(holdout_train_df)} rows")
print(f"Holdout test set: {len(holdout_test_df)} rows")
print(f"\nTest label balance: {holdout_test_df['label'].value_counts(normalize=True).to_dict()}")

# Sanity check: confirm dateline actually stripped
def has_reuters_dateline(text, check_chars=100):
    return bool(re.search(r"\(Reuters\)", str(text)[:check_chars]))

print(f"\nReal articles still containing dateline after stripping: {full_df[full_df['label']=='real']['text'].apply(has_reuters_dateline).mean():.2%}")


Holdout train set: 27001 rows
Holdout test set: 11634 rows

Test label balance: {'real': 0.85808836169847, 'fake': 0.14191163830153}

Real articles still containing dateline after stripping: 0.03%


## 2. TF-IDF Baseline — Combined Test


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

combined_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english", min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

combined_pipeline.fit(holdout_train_df["text"], holdout_train_df["label_id"])

combined_preds = combined_pipeline.predict(holdout_test_df["text"])
combined_acc = accuracy_score(holdout_test_df["label_id"], combined_preds)

print(f"Combined test (subject-holdout + dateline-stripped) TF-IDF Accuracy: {combined_acc:.4f}")
print(classification_report(holdout_test_df["label_id"], combined_preds, target_names=["real", "fake"]))

# Compare against a trivial "always predict majority class" baseline given test set imbalance
majority_baseline = holdout_test_df["label_id"].value_counts(normalize=True).max()
print(f"\nTrivial 'always predict majority class' baseline: {majority_baseline:.4f}")


Combined test (subject-holdout + dateline-stripped) TF-IDF Accuracy: 0.9229
              precision    recall  f1-score   support

        real       0.99      0.92      0.95      9983
        fake       0.66      0.93      0.77      1651

    accuracy                           0.92     11634
   macro avg       0.83      0.93      0.86     11634
weighted avg       0.94      0.92      0.93     11634


Trivial 'always predict majority class' baseline: 0.8581


## 3. DistilBERT — Combined Test


In [3]:
import torch
from transformers import (
    DistilBertTokenizerFast, DistilBertForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.model_selection import train_test_split as sk_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

holdout_train_split, holdout_val_split = sk_split(
    holdout_train_df, test_size=0.1, random_state=42, stratify=holdout_train_df["label_id"]
)

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    return Dataset.from_pandas(df[["text", "label_id"]].rename(columns={"label_id": "labels"}).reset_index(drop=True))

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_ds = to_hf_dataset(holdout_train_split).map(tokenize_fn, batched=True)
val_ds = to_hf_dataset(holdout_val_split).map(tokenize_fn, batched=True)
test_ds = to_hf_dataset(holdout_test_df).map(tokenize_fn, batched=True)

for ds in [train_ds, val_ds, test_ds]:
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

training_args = TrainingArguments(
    output_dir="./distilbert_combined_test",
    num_train_epochs=5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="f1",
    logging_steps=50, report_to="none"
)

trainer = Trainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()


Using device: cuda


Map:   0%|          | 0/24300 [00:00<?, ? examples/s]

Map:   0%|          | 0/2701 [00:00<?, ? examples/s]

Map:   0%|          | 0/11634 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6076, training_loss=0.02147181032613496, metrics={'train_runtime': 1446.9717, 'train_samples_per_second': 83.968, 'train_steps_per_second': 5.249, 'total_flos': 6437915574681600.0, 'train_loss': 0.02147181032613496, 'epoch': 4.0})

In [4]:
test_output = trainer.predict(test_ds)
test_preds = test_output.predictions.argmax(axis=-1)
test_labels = test_output.label_ids

distilbert_combined_acc = accuracy_score(test_labels, test_preds)
print(f"DistilBERT combined test Accuracy: {distilbert_combined_acc:.4f}")
print(classification_report(test_labels, test_preds, target_names=["real", "fake"]))
print(f"\nVs trivial majority-class baseline: {majority_baseline:.4f}")


DistilBERT combined test Accuracy: 0.2191
              precision    recall  f1-score   support

        real       1.00      0.09      0.17      9983
        fake       0.15      1.00      0.27      1651

    accuracy                           0.22     11634
   macro avg       0.58      0.54      0.22     11634
weighted avg       0.88      0.22      0.18     11634


Vs trivial majority-class baseline: 0.8581


## 4. RoBERTa — Combined Test


In [5]:
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

ROBERTA_MODEL_NAME = "roberta-base"
roberta_tokenizer = RobertaTokenizerFast.from_pretrained(ROBERTA_MODEL_NAME)

def roberta_tokenize_fn(batch):
    return roberta_tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

roberta_train_ds = to_hf_dataset(holdout_train_split).map(roberta_tokenize_fn, batched=True)
roberta_val_ds = to_hf_dataset(holdout_val_split).map(roberta_tokenize_fn, batched=True)
roberta_test_ds = to_hf_dataset(holdout_test_df).map(roberta_tokenize_fn, batched=True)

for ds in [roberta_train_ds, roberta_val_ds, roberta_test_ds]:
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

roberta_model = RobertaForSequenceClassification.from_pretrained(ROBERTA_MODEL_NAME, num_labels=2)
roberta_model.to(device)

roberta_training_args = TrainingArguments(
    output_dir="./roberta_combined_test",
    num_train_epochs=5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1, save_total_limit=1,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1",
    logging_steps=50, report_to="none"
)

roberta_trainer = Trainer(
    model=roberta_model, args=roberta_training_args,
    train_dataset=roberta_train_ds, eval_dataset=roberta_val_ds,
    compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

roberta_trainer.train()


Map:   0%|          | 0/24300 [00:00<?, ? examples/s]

Map:   0%|          | 0/2701 [00:00<?, ? examples/s]

Map:   0%|          | 0/11634 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7595, training_loss=0.17316520058692797, metrics={'train_runtime': 3513.1333, 'train_samples_per_second': 34.585, 'train_steps_per_second': 2.162, 'total_flos': 1.598399661312e+16, 'train_loss': 0.17316520058692797, 'epoch': 5.0})

In [6]:
roberta_test_output = roberta_trainer.predict(roberta_test_ds)
roberta_test_preds = roberta_test_output.predictions.argmax(axis=-1)
roberta_test_labels = roberta_test_output.label_ids

roberta_combined_acc = accuracy_score(roberta_test_labels, roberta_test_preds)
print(f"RoBERTa combined test Accuracy: {roberta_combined_acc:.4f}")
print(classification_report(roberta_test_labels, roberta_test_preds, target_names=["real", "fake"]))
print(f"\nVs trivial majority-class baseline: {majority_baseline:.4f}")


RoBERTa combined test Accuracy: 0.8680
              precision    recall  f1-score   support

        real       0.99      0.85      0.92      9983
        fake       0.52      0.97      0.68      1651

    accuracy                           0.87     11634
   macro avg       0.76      0.91      0.80     11634
weighted avg       0.93      0.87      0.88     11634


Vs trivial majority-class baseline: 0.8581


## 5. Final Summary — All Experiments Combined


In [7]:
final_summary = pd.DataFrame([
    {"model": "TF-IDF + Logistic Regression", "combined_test_accuracy": combined_acc},
    {"model": "DistilBERT", "combined_test_accuracy": distilbert_combined_acc},
    {"model": "RoBERTa", "combined_test_accuracy": roberta_combined_acc},
])
final_summary["majority_class_baseline"] = majority_baseline
final_summary["above_trivial_baseline"] = final_summary["combined_test_accuracy"] - majority_baseline

final_summary.to_csv("combined_leakage_test_results.csv", index=False)
final_summary


,model,combined_test_accuracy,majority_class_baseline,above_trivial_baseline
0,TF-IDF + Logistic Regression,0.922898,0.858088,0.064810
1,DistilBERT,0.219099,0.858088,-0.638989
2,RoBERTa,0.867973,0.858088,0.009885
